# Step 4.g — SVM climate suitability, ERA5 and the four AMOC scenarios (EC-Earth3 grid, global)

For each crop, the SVM of 3.g (trained on ERA5 1981–2015 over the MIRCA2000 cropland, with the BIO variant
`CHOSEN_VARIANT` and the radiation `CHOSEN_RAD` of the tests) is applied to **every land cell**:
- **ERA5** (1981–2015): the BIO variables of 1.1.d, the reference climate;
- the **four AMOC scenarios** (2025 / 2050 stabilisation, original and hosing): the delta BIO variables of 1.3.a
  (ERA5 + EC-Earth3 change).

Suitability = the SVM probability (`predict_proba`, 0–1); binary suitability = probability ≥ `THRESHOLD` (0.5, as
before). Suitable area from the area of the EC-Earth3 cells (land cells, whole cell). Climate only: the soil
constraint comes in step 5 / 6.

**Climates outside the training (MESS).** The SVM is trained on the MIRCA2000 cropland only: it has never seen the
climates beyond the edge of today's agriculture (Antarctica, Greenland, central Sahara), and far from the training
cells an RBF SVM gives back its intercept (Antarctica "suitable" for maize). The multivariate environmental
similarity surface (MESS, Elith et al. 2010, *Methods Ecol. Evol.* 1, 330–342) measures, for each cell, how far each
BIO variable is from the values of the training cells of that crop (ERA5 1981–2015 cropland); MESS < 0 = at least one
variable outside the training range ("novel climate"). Those cells are set **not suitable** (`suit_proba` 0); the
SVM alone stays in `suit_proba_svm`, the MESS in `mess` and the variable furthest outside in `mess_var` (the
"most dissimilar variable", MoD). The same reference (ERA5) for the four scenarios, so a scenario climate outside
today's cultivated climates is flagged too.

Outputs in `<out_root>/4_svm_predict/<route>/<variant>_<radiation>/`: `suitability_<crop>.nc`
(`suit_proba`, `suit_bin`, `suit_proba_svm`, `mess`, `mess_var`, dimension `scenario`), `suitable_area_<...>.csv`,
`novel_climate_<...>.csv` and maps.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters: the choices of the tests of 3.g
CHOSEN_VARIANT = None              # test 1
CHOSEN_RAD = None                  # test 2
CELL_AREA_THR = None              # None = route no_threshold (tests 1-3); 0.0005 = route cellareafrac, presence and
                                  # background = area > 0.05 % of the cell land area (test 4); same value in 1.1.d, 3.g, 1.3.a, 4.g
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2015")
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
THRESHOLD = 0.5
assert CHOSEN_VARIANT in ("A_60d", "B_season", "C_crop") and CHOSEN_RAD in ("rad_era5", "no_rad", "rad_fao56"), \
    "set CHOSEN_VARIANT and CHOSEN_RAD to the choices of the tests of 3.g"

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ROUTE = cu.route(CELL_AREA_THR)
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
DELTA = cu.out_dir(CFG, "1_biovariables", f"GCM_delta_{ROUTE}", SET)
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_{ROUTE}", SET)
OUT = cu.out_dir(CFG, "4_svm_predict", ROUTE, SET)

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
CRO = cu.mirca_cropland(CFG, CELL_AREA_THR).values        # training background
# area of the EC-Earth3 cells (Mha): R^2 x dlon x (sin lat_north - sin lat_south)
R = 6371.0
dlon = np.radians(np.diff(GRID["lon_edges"]))
dsin = np.diff(np.sin(np.radians(GRID["lat_edges"])))
AREA_MHA = (R ** 2 * dsin[:, None] * dlon[None, :]) / 1e4
SCEN = {"ERA5": None, **{r: r for r in RUNS}}
print(f"set {SET} | models {MODELS}\noutputs {OUT}")

## 1. Prediction

In [ ]:
def bio_file(scenario, crop):
    if scenario == "ERA5":
        return ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc"
    return DELTA / f"BIOvariables_delta_{scenario}_{crop}.nc"


def mess(X, ref):
    # MESS (Elith et al. 2010): f = % of the reference cells below the value; similarity 2f (f <= 50), 2(100 - f)
    # (50 < f < 100), and below the minimum / above the maximum the distance to it in % of the range (negative);
    # MESS = the smallest similarity over the variables, MoD = that variable
    lo, hi = ref.min(axis=0), ref.max(axis=0)
    rng = np.where(hi > lo, hi - lo, 1.0)
    S = np.empty_like(X)
    for k in range(X.shape[1]):
        f = 100 * np.searchsorted(np.sort(ref[:, k]), X[:, k], side="left") / len(ref)
        S[:, k] = np.where(f == 0, 100 * (X[:, k] - lo[k]) / rng[k],
                  np.where(f <= 50, 2 * f,
                  np.where(f < 100, 2 * (100 - f), 100 * (hi[k] - X[:, k]) / rng[k])))
    return S.min(axis=1), S.argmin(axis=1)


def predict(bundle, path, ref):
    with xr.open_dataset(path) as ds:
        X = np.stack([ds[c].values for c in bundle["bio_cols"]], axis=-1).reshape(-1, len(bundle["bio_cols"]))
    ok = np.isfinite(X).all(axis=1) & LAND.ravel()
    p, m, v = np.full(X.shape[0], np.nan), np.full(X.shape[0], np.nan), np.full(X.shape[0], -1)
    p[ok] = bundle["model"].predict_proba(bundle["scaler"].transform(X[ok]))[:, 1]
    m[ok], v[ok] = mess(X[ok], ref)
    return p.reshape(LAND.shape), m.reshape(LAND.shape), v.reshape(LAND.shape)


rows, novel_rows = [], []
for crop in CROPS:
    bundle = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    with xr.open_dataset(bio_file("ERA5", crop)) as ds:  # training cells: the climates the SVM has seen (from the
        REF = np.stack([ds[c].values[ii, jj] for c in bundle["bio_cols"]], axis=-1)   # netCDF, not the rounded csv)
    REF = REF[np.isfinite(REF).all(axis=1)]
    R3 = {s: predict(bundle, bio_file(s, crop), REF) for s in SCEN}
    stack = lambda k: xr.DataArray(np.stack([R3[s][k] for s in SCEN]), dims=("scenario", "lat", "lon"),
                                   coords={"scenario": list(SCEN), "lat": GRID["lat"], "lon": GRID["lon"]})
    p_svm, mss, mvar = stack(0), stack(1), stack(2)
    novel = mss < 0
    proba = xr.where(novel, 0.0, p_svm)                   # novel climate -> not suitable (NaN stays NaN)
    suit = xr.where(proba.notnull(), (proba >= THRESHOLD).astype("float32"), np.nan)
    xr.Dataset({"suit_proba": proba.astype("float32").assign_attrs(long_name="suitability probability, 0 where MESS < 0"),
                "suit_bin": suit.assign_attrs(long_name="binary suitability", threshold=THRESHOLD),
                "suit_proba_svm": p_svm.astype("float32").assign_attrs(long_name="SVM probability, no MESS"),
                "mess": mss.astype("float32").assign_attrs(long_name="MESS (Elith et al. 2010), reference ERA5 training cells",
                                                           note="< 0: at least one BIO variable outside the training range"),
                "mess_var": mvar.astype("int16").assign_attrs(long_name="most dissimilar variable (index in bio_cols)",
                                                              bio_cols=", ".join(bundle["bio_cols"]), missing=-1)},
               attrs={"crop": crop, "set": SET, "model": f"one-pass SVM, cost {bundle['best_cost']}, gamma {bundle['best_gamma']}",
                      "bio_cols": ", ".join(bundle["bio_cols"])}).to_netcdf(OUT / f"suitability_{crop}.nc")
    # in-sample check on the training cells (not a validation: the cross-validation of 3.g is)
    auc_in = roc_auc_score(tr.label.values, proba.sel(scenario="ERA5").values[ii, jj])
    era5_area = float(np.nansum(AREA_MHA * (suit.sel(scenario="ERA5").values == 1)))
    for s in SCEN:
        b = suit.sel(scenario=s).values == 1
        a = float(np.nansum(AREA_MHA * b))
        nv = novel.sel(scenario=s).values
        rows.append({"crop": crop, "scenario": s, "suitable area (Mha)": a,
                     "change vs ERA5 (%)": 100 * (a - era5_area) / era5_area if s != "ERA5" else 0.0,
                     "on MIRCA cropland (Mha)": float(np.nansum(AREA_MHA * (b & CRO))),
                     "SVM alone, no MESS (Mha)": float(np.nansum(AREA_MHA * (p_svm.sel(scenario=s).values >= THRESHOLD))),
                     "novel climate (Mha)": float(np.nansum(AREA_MHA * nv)),
                     "novel on MIRCA cropland (Mha)": float(np.nansum(AREA_MHA * (nv & CRO))),
                     "mean probability on cropland": float(np.nanmean(proba.sel(scenario=s).values[CRO])),
                     "in-sample AUC (ERA5)": auc_in if s == "ERA5" else np.nan})
        # which variable puts the cells outside the training range
        v = mvar.sel(scenario=s).values
        for k, c in enumerate(bundle["bio_cols"]):
            sel = nv & (v == k)
            if sel.any():
                novel_rows.append({"crop": crop, "scenario": s, "variable": c, "novel area (Mha)": float(np.nansum(AREA_MHA * sel)),
                                   "of which on MIRCA cropland (Mha)": float(np.nansum(AREA_MHA * (sel & CRO)))})
    print(f"{crop}: ERA5 {era5_area:.1f} Mha suitable, in-sample AUC {auc_in:.3f}, "
          f"novel climate (ERA5) {rows[-len(SCEN)]['novel climate (Mha)']:.0f} Mha", flush=True)
area = pd.DataFrame(rows).set_index(["crop", "scenario"])
area.to_csv(OUT / f"suitable_area_{SET}.csv")
novel_tab = pd.DataFrame(novel_rows, columns=["crop", "scenario", "variable", "novel area (Mha)",
                                              "of which on MIRCA cropland (Mha)"]).set_index(["crop", "scenario", "variable"])
novel_tab.to_csv(OUT / f"novel_climate_{SET}.csv")
with pd.option_context("display.max_rows", 200, "display.width", 200):
    display(area.round(2))
    print("novel climate (MESS < 0) by most dissimilar variable"); display(novel_tab.round(2))

## 2. Effect of the AMOC collapse: hosing − original, per stabilisation level

In [ ]:
rows = []
for crop in CROPS:
    for hos, orig in PAIRS:
        a_h, a_o = area.loc[(crop, hos), "suitable area (Mha)"], area.loc[(crop, orig), "suitable area (Mha)"]
        rows.append({"crop": crop, "pair": f"{hos} - {orig}", "area change (Mha)": a_h - a_o,
                     "area change (%)": 100 * (a_h - a_o) / a_o if a_o else np.nan})
amoc = pd.DataFrame(rows).set_index(["crop", "pair"])
amoc.to_csv(OUT / f"amoc_effect_{SET}.csv")
display(amoc.round(2))

for crop in CROPS:
    ds = xr.open_dataset(OUT / f"suitability_{crop}.nc")
    fig, axes = plt.subplots(2, 3, figsize=(20, 7))
    for ax, s in zip(axes[0], ["ERA5", PAIRS[0][1], PAIRS[1][1]]):
        ds.suit_proba.sel(scenario=s).plot(ax=ax, vmin=0, vmax=1, cmap="YlGn"); ax.set_title(f"{crop}: {s}", fontsize=9)
    ds.mess.sel(scenario="ERA5").clip(-100, 100).plot(ax=axes[1, 0], vmin=-100, vmax=100, cmap="PuOr")
    axes[1, 0].set_title(f"{crop}: MESS ERA5 (< 0 = novel climate, set not suitable)", fontsize=9)
    for ax, (hos, orig) in zip(axes[1, 1:], PAIRS):
        (ds.suit_proba.sel(scenario=hos) - ds.suit_proba.sel(scenario=orig)).plot(ax=ax, vmin=-0.5, vmax=0.5, cmap="RdBu")
        ax.set_title(f"{hos} - {orig}", fontsize=9)
    for ax in axes.ravel():
        ax.set_xlabel(""); ax.set_ylabel("")
    plt.tight_layout(); plt.savefig(OUT / f"suitability_maps_{SET}_{crop}.png", dpi=80); plt.show()